HI-Union PPI Database:

Step 1: Converting dataset to a sanitised version for faster access and lower memory usage.

In [6]:
import csv
import pandas as pd
def extract_first_two_columns(input_path, output_path):
    with open(input_path, "r", encoding="utf-8") as infile, \
         open(output_path, "w", newline="", encoding="utf-8") as outfile:

        reader = csv.reader(infile, delimiter="\t")
        writer = csv.writer(outfile)

        for row in reader:
            if not row:
                continue
            interactor_a, interactor_b = row[0], row[1]
        
            writer.writerow([interactor_a.split(':', 1)[1], interactor_b.split(':',1)[1], row[14].split(':')[1] if row[14] != '-' else row[14]])

extract_first_two_columns("HI-union.psi", "sanitised.csv")
df = pd.read_csv("sanitised.csv", header=None)
interactors = df.iloc[:, [0, 1,2]]
interactors.columns = ["Uniprot_A", "Uniprot_B", "author_score"]

interactors.to_csv("sanitised.csv", index=False, header=True)

In [31]:
import pandas as pd
import networkx as nx
from collections import defaultdict

def create_graph(path_to_csv):
    dataframe = pd.read_csv(path_to_csv, sep='\t')
    G = nx.Graph()
    
    graph = defaultdict(set)
    
    for gene_a, gene_b in zip(dataframe['Uniprot_A'],dataframe['Uniprot_B']):
        G.add_nodes_from([gene_a,gene_b])
        if pd.isna(gene_a) or pd.isna(gene_b):
            continue
        if gene_a == gene_b:
            continue
        graph[gene_a].add(gene_b)
        graph[gene_b].add(gene_a)
        G.add_edge(gene_a,gene_b)
    return graph, G

In [36]:
from collections import deque
def BFS(source, targets, graph):
    if source in targets:
        return 0
    visited = set(source)
    queue = deque([(source,0)])
    while queue:
        node, dist = queue.popleft()
        for neighbour in list(graph[node]):
            if neighbour in visited:
                continue
            if neighbour in targets:
                return dist+1
            queue.append((neighbour, dist+1))
            visited.add(neighbour)
    return None

source = 'Q9UK80-1'
targets = ['P51116','O15529']
graph, _ = create_graph('sanitised.tsv')

BFS(source,targets,graph)

3